In [1]:
!pip install -q tensorflow scikit-learn seaborn matplotlib

from google.colab import drive
drive.mount('/content/drive')
print('✅ Packages installed and Drive mounted')

Mounted at /content/drive
✅ Packages installed and Drive mounted


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ─── YOUR DATA IS HERE ───────────────────────────────────────────────────────
# My Drive > AI project > Notebooks
#   ├── Car/    ← car images
#   └── Bike/   ← bike images
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/AI project/Notebooks'

# Results will be saved here (created automatically)
SAVE_DIR = '/content/drive/MyDrive/AI project/Notebooks/results'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Verify the folder structure ───────────────────────────────────────────────
print('Checking folder structure...')
found_classes = []
for item in sorted(os.listdir(DATA_DIR)):
    full = os.path.join(DATA_DIR, item)
    if os.path.isdir(full) and item != 'results':
        images = [f for f in os.listdir(full)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        print(f'  📁 {item}/  →  {len(images)} images')
        found_classes.append(item)

if len(found_classes) == 2:
    print(f'\n✅ Found 2 classes: {found_classes}  — ready to train!')
else:
    print(f'\n⚠️  Expected 2 class folders (Car, Bike). Found: {found_classes}')
    print('    Make sure the folders are named exactly  Car  and  Bike')

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16

# Augmentation only on training data
train_datagen = ImageDataGenerator(
    rescale          = 1.0 / 255,
    rotation_range   = 10,
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    horizontal_flip  = True,
    zoom_range       = 0.1,
    validation_split = 0.2       # 80% train, 20% validation
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    subset      = 'training',
    shuffle     = True,
    seed        = 42,
    classes     = ['Car', 'Bike']   # explicit order: index 0=Car, 1=Bike
)

val_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    subset      = 'validation',
    shuffle     = False,
    seed        = 42,
    classes     = ['Car', 'Bike']
)

print('Class indices :', train_gen.class_indices)
print(f'Training images: {train_gen.samples}')
print(f'Validation images: {val_gen.samples}')

In [ ]:
NUM_CLASSES = 2  # Car, Bike

# Load ResNet50 with ImageNet weights, remove the top classification layer
base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False   # freeze — only train our new head layers

# Add custom classification head
x = base.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base.input, outputs=outputs)

model.compile(
    optimizer = 'adam',
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

print('✅ ResNet50 model built')
print(f'Total params      : {model.count_params():,}')
print(f'Trainable params  : {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

In [ ]:
best_model_path = os.path.join(SAVE_DIR, 'best_model.h5')

callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        best_model_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

history = model.fit(
    train_gen,
    epochs          = 10,
    validation_data = val_gen,
    callbacks       = callbacks,
    verbose         = 1
)

print('\n✅ Training complete')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train', marker='o')
ax1.plot(history.history['val_accuracy'], label='Val',   marker='o')
ax1.set_title('Accuracy per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train', marker='o')
ax2.plot(history.history['val_loss'], label='Val',   marker='o')
ax2.set_title('Loss per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_history.png'), dpi=150)
plt.show()
print('✅ Training history saved')

In [ ]:
val_gen.reset()
preds        = model.predict(val_gen, verbose=1)
pred_classes = np.argmax(preds, axis=1)
true_classes = val_gen.classes
class_names  = list(val_gen.class_indices.keys())   # ['Car', 'Bike']

# Confusion matrix
cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Oranges',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Car vs Bike')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

print('\n=== Classification Report ===')
print(classification_report(true_classes, pred_classes, target_names=class_names))

In [ ]:
# Save full model
final_model_path = os.path.join(SAVE_DIR, 'car_bike_resnet50.h5')
model.save(final_model_path)

# Save class index map so the website knows which index = Car/Bike
class_map_path = os.path.join(SAVE_DIR, 'class_names.json')
with open(class_map_path, 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)

print('✅ Files saved to Drive:')
print(f'   Model      : {final_model_path}')
print(f'   Best model : {best_model_path}')
print(f'   Class map  : {class_map_path}')

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

# ── Pick any image from your Drive to test ───────────────────────────────────
# Change this path to any car or bike image you have
TEST_IMAGE = '/content/drive/MyDrive/AI project/Notebooks/Car/car_001.jpg'
# ─────────────────────────────────────────────────────────────────────────────

if os.path.exists(TEST_IMAGE):
    img  = keras_image.load_img(TEST_IMAGE, target_size=(224, 224))
    arr  = keras_image.img_to_array(img) / 255.0
    arr  = np.expand_dims(arr, axis=0)

    pred = model.predict(arr, verbose=0)
    idx  = np.argmax(pred[0])
    conf = pred[0][idx] * 100
    label = class_names[idx]

    plt.figure(figsize=(5, 4))
    plt.imshow(img)
    plt.axis('off')
    icon = '🚗' if label == 'Car' else '🏍️'
    plt.title(f'{icon}  Prediction: {label}  ({conf:.1f}%)', fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f'Result: {label} with {conf:.1f}% confidence')
else:
    print(f'⚠️  Image not found at: {TEST_IMAGE}')
    print('    Change TEST_IMAGE path to any .jpg in your Car/ or Bike/ folder')

---
## 🌐 Connecting to Your Website

The trained model connects to `index.html` via the **Roboflow API** (easiest) or **TensorFlow.js**.

### Option A — Roboflow (Recommended)
1. Go to https://app.roboflow.com → New Project → `car-vs-bike` → Classification
2. Upload your `Car/` and `Bike/` folders, label them `car` and `bike`
3. Generate version → Train (25 epochs, ImageNet checkpoint)
4. After training → copy your **API Key** and **Model ID** (`car-vs-bike/1`)
5. Open `index.html`, find the CONFIG section and paste:
```javascript
const ROBOFLOW_API_KEY = 'your-key-here';
const ROBOFLOW_MODEL_ID = 'car-vs-bike/1';
```

### Option B — TensorFlow.js (use the model you trained above)
```bash
# Run in Colab
!pip install tensorflowjs
!tensorflowjs_converter --input_format keras \
    '/content/drive/MyDrive/AI project/Notebooks/results/car_bike_resnet50.h5' \
    '/content/drive/MyDrive/AI project/Notebooks/results/tfjs_model'
```
Then host the `tfjs_model/` folder and load it in the website with `@tensorflow/tfjs`.

> **Easiest:** Use Roboflow — no hosting setup needed.